# HDBSCAN Clustering for Betacov sequences

In [ ]:
version_dir = "betacov_base"
version_prefix = "BETACOV.CLS-embedded"

In [2]:
import os
import pandas as pd

data_dir = "/panfs/biopan03/prime_ml/prime/data/betacov"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/parquets"

parquet_file = os.path.join(data_dir, "betacoronavirus_seq_CLS-embedded.parquet")
embedded_df = pd.read_parquet(parquet_file, engine='fastparquet')

print(embedded_df['variant'].value_counts())
display(embedded_df)

variant
Bat Virus         33
MERSV             21
SARS-CoV-1        15
SARS-CoV-2         3
Pangolin Virus     2
Hibecovirus        1
Name: count, dtype: int64


,seq_id,variant,host,embedding
0,MN996532,Bat Virus,Bat,"[0.009531335, 0.68848395, 0.12244794, 0.202172..."
1,MG772934,Bat Virus,Bat,"[-0.0078075435, 0.76730734, 0.1585119, 0.16685..."
2,MG772933,Bat Virus,Bat,"[0.00084287755, 0.7605357, 0.16206563, 0.17404..."
3,KT444582,Bat Virus,Bat,"[0.022269862, 0.735213, 0.054602887, 0.2219388..."
4,KY417146,Bat Virus,Bat,"[0.027052812, 0.7307338, 0.09586543, 0.2022184..."
...,...,...,...,...
70,AKN24803.1,MERSV,Human,"[0.09266137, 0.54673946, 0.32283795, 0.2776477..."
71,ASU90527.1,MERSV,Camel,"[0.09696581, 0.5694871, 0.31734318, 0.27250516..."
72,AKL59401.1,MERSV,Human,"[0.09163268, 0.5574062, 0.32339773, 0.2764253,..."
73,ASY99842.1,MERSV,Human,"[0.091899104, 0.5741013, 0.3097003, 0.27296144..."


In [ ]:
version_dir = "betacov_finetuned"
version_prefix = "BETACOV.from-esm-mlm_CLS-embedded"

In [4]:
import os
import pandas as pd

data_dir = "/panfs/biopan03/prime_ml/prime/data/betacov"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/parquets"

parquet_file = os.path.join(data_dir, "betacoronavirus_seq_from-esm-mlm_CLS-embedded.parquet")
embedded_df = pd.read_parquet(parquet_file, engine='fastparquet')

print(embedded_df['variant'].value_counts())
display(embedded_df)

variant
Bat Virus         33
MERSV             21
SARS-CoV-1        15
SARS-CoV-2         3
Pangolin Virus     2
Hibecovirus        1
Name: count, dtype: int64


,seq_id,variant,host,embedding
0,MN996532,Bat Virus,Bat,"[-0.19183712, 0.45011306, 0.49967805, 0.756774..."
1,MG772934,Bat Virus,Bat,"[-0.30005515, 0.65058917, 0.6550717, 0.7039596..."
2,MG772933,Bat Virus,Bat,"[-0.28308988, 0.6491581, 0.66039366, 0.7101965..."
3,KT444582,Bat Virus,Bat,"[-0.23724484, 0.4270615, 0.4050742, 0.73820573..."
4,KY417146,Bat Virus,Bat,"[-0.22176376, 0.4007309, 0.45212156, 0.72232, ..."
...,...,...,...,...
70,AKN24803.1,MERSV,Human,"[-0.25705302, 0.6160833, 0.453325, 0.57901525,..."
71,ASU90527.1,MERSV,Camel,"[-0.2506442, 0.61880815, 0.44428322, 0.5544402..."
72,AKL59401.1,MERSV,Human,"[-0.26611528, 0.62681794, 0.4465537, 0.5578939..."
73,ASY99842.1,MERSV,Human,"[-0.25976172, 0.5912259, 0.4469077, 0.55536854..."


In [6]:
import os
from sklearn.model_selection import ParameterGrid
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

import cupy as cp
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
from cuml.preprocessing import LabelEncoder
from cuml.metrics.cluster import adjusted_rand_score, silhouette_score

def umap_hdbscan_parameter_sweep(parquet_file, embedded_df, n_neighbors_list, min_dist_list, min_samples_list, min_cluster_size_list):
    umap_grid = {
        "n_neighbors": n_neighbors_list,
        "min_dist": min_dist_list,
    }
    umap_param_list = list(ParameterGrid(umap_grid))

    hdbscan_grid = {
        "min_samples": min_samples_list,
        "min_cluster_size": min_cluster_size_list,
    }
    hdbscan_param_list = list(ParameterGrid(hdbscan_grid))

    all_results = []

    # CPU -> GPU
    embedding_matrix = np.vstack(embedded_df["embedding"].values).astype(np.float32)
    X_gpu = cp.asarray(embedding_matrix)
    y_true = LabelEncoder().fit_transform(embedded_df["variant"].values)
    y_true_gpu = cp.asarray(y_true)

    # Parameter Sweep (UMAP)
    for umap_params in tqdm(umap_param_list, desc="UMAP sweep"):
        reducer = UMAP(
            n_neighbors=umap_params["n_neighbors"],
            min_dist=umap_params["min_dist"],
            metric="cosine",
            random_state=42,
        )
        X_red = reducer.fit_transform(X_gpu)
        X_red_cpu = cp.asnumpy(X_red)

        umap_df = pd.DataFrame({
            'UMAP 1': X_red_cpu[:, 0],
            'UMAP 2': X_red_cpu[:, 1],
            'Seq ID': embedded_df['seq_id'].values,
            'Variant': embedded_df['variant'].values,
            'Host': embedded_df['host'].values,
        })
        save_as = parquet_file.replace(".parquet", f".umap-nneighbors{umap_params["n_neighbors"]}_mindist{umap_params["min_dist"]}.parquet")
        umap_df.to_parquet(save_as, engine='fastparquet')

        # Parameter Sweep (HDBSCAN)
        for hdbscan_params in tqdm(hdbscan_param_list, leave=False):
            clusterer = HDBSCAN(
                min_samples=hdbscan_params["min_samples"],
                min_cluster_size=hdbscan_params["min_cluster_size"],
            )

            labels = clusterer.fit_predict(X_red)
            ari = float(adjusted_rand_score(y_true_gpu, labels))
            noise_fraction = float(cp.mean(labels == -1))

            non_noise_mask = labels != -1
            valid_cluster_labels = labels[non_noise_mask]
            
            n_non_noise_samples = int(cp.sum(non_noise_mask))
            n_non_noise_clusters = int(cp.unique(valid_cluster_labels).size)

            # silhouette needs at least two clusters AND
            # avoids the case where every non-noise point is its own cluster
            if n_non_noise_clusters > 1 and n_non_noise_samples > n_non_noise_clusters:
                silhouette_avg = float(silhouette_score(X_red[non_noise_mask], valid_cluster_labels,  metric="cosine"))
            else:
                silhouette_avg = np.nan            

            all_results.append({
                "n_neighbors": umap_params["n_neighbors"],
                "min_dist": umap_params["min_dist"],
                "min_samples": hdbscan_params["min_samples"],
                "min_cluster_size": hdbscan_params["min_cluster_size"],
                "ari": ari,
                "noise_fraction": noise_fraction,
                "silhouette": silhouette_avg,
                "n_clusters": n_non_noise_clusters,
            })
            
        # cleanup
        del X_red
        cp._default_memory_pool.free_all_blocks()

    # Final results of both parameter sweeps
    full_results_df = pd.DataFrame(all_results) # All results
    return full_results_df

/panfs/biopan03/prime_ml/miniconda3/envs/blackwell_prime/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
version_dir = "betacov_base"
version_prefix = "BETACOV.CLS-embedded"

In [8]:
%%time

data_dir = "/panfs/biopan03/prime_ml/prime/data/betacov"
sweep_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/param_sweeps"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/parquets"

umap_grid = {
    "n_neighbors": list(range(2,74)),
    "min_dist": [0.0, 0.1, 0.15, 0.25, 0.5, 0.75, 1.0],
}

hdbscan_grid = {
    "min_samples": range(1, 26),
    "min_cluster_size": range(2, 26),
}

parquet_file = os.path.join(data_dir, "betacoronavirus_seq_CLS-embedded.parquet")
embedded_df = pd.read_parquet(parquet_file, engine="fastparquet")

parquet_file_name = os.path.join(parquet_dir, version_dir, f"{version_prefix}.parquet")

full_results_umap_df = umap_hdbscan_parameter_sweep(
    os.path.join(parquet_dir, version_dir, parquet_file_name),
    embedded_df, 
    umap_grid["n_neighbors"],
    umap_grid["min_dist"],
    hdbscan_grid["min_samples"],
    hdbscan_grid["min_cluster_size"],
)

full_results_umap_df.to_csv(os.path.join(sweep_dir, version_dir, f"{version_prefix}-umap_hdbscan-parameter_sweep.csv"), index=False)

UMAP sweep: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 504/504 [2:26:01<00:00, 17.38s/it]


CPU times: user 2h 24min 42s, sys: 57.5 s, total: 2h 25min 39s
Wall time: 2h 26min 3s


In [9]:
version_dir = "betacov_finetuned"
version_prefix = "BETACOV.from-esm-mlm_CLS-embedded"

In [10]:
%%time

data_dir = "/panfs/biopan03/prime_ml/prime/data/betacov"
sweep_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/param_sweeps"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/parquets"

umap_grid = {
    "n_neighbors": list(range(2,74)),
    "min_dist": [0.0, 0.1, 0.15, 0.25, 0.5, 0.75, 1.0],
}

hdbscan_grid = {
    "min_samples": range(1, 26),
    "min_cluster_size": range(2, 26),
}

parquet_file = os.path.join(data_dir, "betacoronavirus_seq_from-esm-mlm_CLS-embedded.parquet")
embedded_df = pd.read_parquet(parquet_file, engine="fastparquet")

parquet_file_name = os.path.join(parquet_dir, version_dir, f"{version_prefix}.parquet")

full_results_umap_df = umap_hdbscan_parameter_sweep(
    os.path.join(parquet_dir, version_dir, parquet_file_name),
    embedded_df, 
    umap_grid["n_neighbors"],
    umap_grid["min_dist"],
    hdbscan_grid["min_samples"],
    hdbscan_grid["min_cluster_size"],
)

full_results_umap_df.to_csv(os.path.join(sweep_dir, version_dir, f"{version_prefix}-umap_hdbscan-parameter_sweep.csv"), index=False)

UMAP sweep: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 504/504 [23:54<00:00,  2.85s/it]


CPU times: user 23min 22s, sys: 23.1 s, total: 23min 46s
Wall time: 23min 55s


In [11]:
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden to 1 by setting random_state")

from sklearn.model_selection import ParameterGrid
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

import cupy as cp
from cuml.manifold import TSNE
from cuml.cluster import HDBSCAN
from cuml.preprocessing import LabelEncoder
from cuml.metrics.cluster import adjusted_rand_score, silhouette_score

def tsne_hdbscan_parameter_sweep(parquet_file, embedded_df, perplexity_list, min_samples_list, min_cluster_size_list):
    hdbscan_grid = {
        "min_samples": min_samples_list,
        "min_cluster_size": min_cluster_size_list,
    }
    hdbscan_param_list = list(ParameterGrid(hdbscan_grid))

    all_results = []

    # CPU -> GPU
    embedding_matrix = np.vstack(embedded_df["embedding"].values).astype(np.float32)
    X_gpu = cp.asarray(embedding_matrix)
    y_true = LabelEncoder().fit_transform(embedded_df["variant"].values)
    y_true_gpu = cp.asarray(y_true)

    n_samples = X_gpu.shape[0]
    early_exaggeration = 24.0 if n_samples > 10000 else 12.0 # Default 12.0
    learning_rate = max(n_samples / early_exaggeration, 200) # Default: 200

    # Parameter Sweep (TSNE)
    for perp in tqdm(perplexity_list, desc="TSNE sweep"):
        save_as = parquet_file.replace(".parquet", f".tsne-perplexity{perp}.parquet")

        if os.path.exists(save_as):
            print(f"Found existing t-SNE file, loading: {save_as}")
            tsne_df = pd.read_parquet(save_as, engine="fastparquet")

            X_red_cpu = tsne_df[["t-SNE component 1", "t-SNE component 2"]].values.astype(np.float32)
            X_red = cp.asarray(X_red_cpu)

        else:
            reducer = TSNE(
                n_components=2,
                perplexity=perp,
                n_neighbors=3 * perp,   # recommended by cuml, caps at 1023
                init="pca",             # like openTSNE
                metric="cosine",
                method="fft",
                random_state=42,
                early_exaggeration=early_exaggeration,
                learning_rate=learning_rate,
                learning_rate_method=None,
            )
            X_red = reducer.fit_transform(X_gpu)
            X_red_cpu = cp.asnumpy(X_red)

            # Save 
            tsne_df = pd.DataFrame({
                't-SNE component 1': X_red_cpu[:, 0],
                't-SNE component 2': X_red_cpu[:, 1],
                'Seq ID': embedded_df['seq_id'].values,
                'Variant': embedded_df['variant'].values,
                'Host': embedded_df['host'].values,
            })
            
            tsne_df.to_parquet(save_as, engine='fastparquet')

        # Parameter Sweep (HDBSCAN)
        for hdbscan_params in tqdm(hdbscan_param_list, leave=False):
            clusterer = HDBSCAN(
                min_samples=hdbscan_params["min_samples"],
                min_cluster_size=hdbscan_params["min_cluster_size"],
            )

            labels = clusterer.fit_predict(X_red)
            ari = float(adjusted_rand_score(y_true_gpu, labels))
            noise_fraction = float(cp.mean(labels == -1))

            non_noise_mask = labels != -1
            valid_cluster_labels = labels[non_noise_mask]
            
            n_non_noise_samples = int(cp.sum(non_noise_mask))
            n_non_noise_clusters = int(cp.unique(valid_cluster_labels).size)

            # silhouette needs at least two clusters AND
            # avoids the case where every non-noise point is its own cluster
            if n_non_noise_clusters > 1 and n_non_noise_samples > n_non_noise_clusters:
                silhouette_avg = float(silhouette_score(X_red[non_noise_mask], valid_cluster_labels,  metric="cosine"))
            else:
                silhouette_avg = np.nan                

            all_results.append({
                "perplexity": perp,
                "min_samples": hdbscan_params["min_samples"],
                "min_cluster_size": hdbscan_params["min_cluster_size"],
                "ari": ari,
                "noise_fraction": noise_fraction,
                "silhouette": silhouette_avg,
                "n_clusters": n_non_noise_clusters,
            })
            
        # cleanup
        del X_red
        cp._default_memory_pool.free_all_blocks()

    # Final results of both parameter sweeps
    full_results_df = pd.DataFrame(all_results) # All results
    return full_results_df

In [12]:
version_dir = "betacov_base"
version_prefix = "BETACOV.CLS-embedded"

In [13]:
%%time

data_dir = "/panfs/biopan03/prime_ml/prime/data/betacov"
sweep_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/param_sweeps"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/parquets"

n_samples = len(embedded_df)
max_perplexity = max(2, (n_samples - 1) // 3)
perplexity_list = range(2, max_perplexity + 1)

hdbscan_grid = {
    "min_samples": range(1, 26),
    "min_cluster_size": range(2, 26),
}

parquet_file = os.path.join(data_dir, "betacoronavirus_seq_CLS-embedded.parquet")
embedded_df = pd.read_parquet(parquet_file, engine="fastparquet")

parquet_file_name = os.path.join(parquet_dir, version_dir, f"{version_prefix}.parquet")

full_results_tsne_df = tsne_hdbscan_parameter_sweep(
    os.path.join(parquet_dir, version_dir, parquet_file_name),
    embedded_df, 
    perplexity_list,
    hdbscan_grid["min_samples"],
    hdbscan_grid["min_cluster_size"],
)

full_results_tsne_df.to_csv(os.path.join(sweep_dir, version_dir, f"{version_prefix}-tsne_hdbscan-parameter_sweep.csv"), index=False)

TSNE sweep:   0%|                                                                                                                                                                                                                                                                                                                                             | 0/23 [00:00<?, ?it/s]

[2026-04-28 06:40:13.708] [CUML] [warning] Perplexity should be within ranges (5, 50). Your results might be a bit strange...


TSNE sweep:   4%|██████████████▏                                                                                                                                                                                                                                                                                                                      | 1/23 [00:02<00:59,  2.70s/it]

[2026-04-28 06:40:16.400] [CUML] [warning] Perplexity should be within ranges (5, 50). Your results might be a bit strange...


TSNE sweep:   9%|████████████████████████████▎                                                                                                                                                                                                                                                                                                        | 2/23 [00:04<00:47,  2.28s/it]

[2026-04-28 06:40:18.378] [CUML] [warning] Perplexity should be within ranges (5, 50). Your results might be a bit strange...


TSNE sweep: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [01:05<00:00,  2.84s/it]


CPU times: user 1min 3s, sys: 261 ms, total: 1min 4s
Wall time: 1min 5s


In [14]:
version_dir = "betacov_finetuned"
version_prefix = "BETACOV.from-esm-mlm_CLS-embedded"

In [15]:
%%time

data_dir = "/panfs/biopan03/prime_ml/prime/data/betacov"
sweep_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/param_sweeps"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/betacov/parquets"

n_samples = len(embedded_df)
max_perplexity = max(2, (n_samples - 1) // 3)
perplexity_list = range(2, max_perplexity + 1)

hdbscan_grid = {
    "min_samples": range(1, 26),
    "min_cluster_size": range(2, 26),
}

parquet_file = os.path.join(data_dir, "betacoronavirus_seq_from-esm-mlm_CLS-embedded.parquet")
embedded_df = pd.read_parquet(parquet_file, engine="fastparquet")

parquet_file_name = os.path.join(parquet_dir, version_dir, f"{version_prefix}.parquet")

full_results_tsne_df = tsne_hdbscan_parameter_sweep(
    os.path.join(parquet_dir, version_dir, parquet_file_name),
    embedded_df, 
    perplexity_list,
    hdbscan_grid["min_samples"],
    hdbscan_grid["min_cluster_size"],
)

full_results_tsne_df.to_csv(os.path.join(sweep_dir, version_dir, f"{version_prefix}-tsne_hdbscan-parameter_sweep.csv"), index=False)

TSNE sweep:   0%|                                                                                                                                                                                                                                                                                                                                             | 0/23 [00:00<?, ?it/s]

[2026-04-28 06:41:18.988] [CUML] [warning] Perplexity should be within ranges (5, 50). Your results might be a bit strange...


TSNE sweep:   4%|██████████████▏                                                                                                                                                                                                                                                                                                                      | 1/23 [00:02<00:44,  2.02s/it]

[2026-04-28 06:41:21.005] [CUML] [warning] Perplexity should be within ranges (5, 50). Your results might be a bit strange...


TSNE sweep:   9%|████████████████████████████▎                                                                                                                                                                                                                                                                                                        | 2/23 [00:03<00:41,  1.99s/it]

[2026-04-28 06:41:22.980] [CUML] [warning] Perplexity should be within ranges (5, 50). Your results might be a bit strange...


TSNE sweep: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [01:00<00:00,  2.64s/it]


CPU times: user 1min, sys: 212 ms, total: 1min
Wall time: 1min
